# Example 12.7 / 1: Solve a Linear Bilevel Instance

Companion notebook for **12.7 Bilevel Optimization** in the AMPL lab Documentation, section "Solve a Linear Bilevel Instance". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve a Linear Bilevel Instance

The leader chooses $0 \le x \le 6$ to minimize $x + 2y$. The follower maximizes $y$ subject to
$y \le 1 + x$, $y \le 6 - x$, $y \ge 0$, so its response is $y = \min(1 + x, 6 - x)$. Written as
$\min -y$, the follower's Lagrangian gives stationarity $\mu_1 + \mu_2 - \nu = 1$ with
multipliers $\mu_1, \mu_2$ for the two limits and $\nu$ for $y \ge 0$. Each complementarity pair
is linearized with a binary, a dual bound and a slack bound; the slack bound 7 is valid because
$1 + x - y \le 7$ and $6 - x - y \le 6$.

The run solves,
repeats with a dual bound that is too small, then fixes $x = 0$ and solves the follower alone.
Declaring `problem Follower` uses `ampl.eval` because amplpy has no method for `problem`
statements; `fix()` and `value()` are amplpy methods.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param dual_bound > 0;                   # assumed bound on follower multipliers
param slack_bound > 0;                  # valid bound on follower slacks

var X >= 0 <= 6;                        # leader decision
var Y >= 0;                             # follower response

# Follower multipliers and active-side indicators.
var Mu1 >= 0 <= dual_bound;
var Mu2 >= 0 <= dual_bound;             # multiplier of FollowerLimit2
var Nu >= 0 <= dual_bound;              # multiplier of Y >= 0
var Active1 binary;                     # 1 = FollowerLimit1 binding
var Active2 binary;                     # 1 = FollowerLimit2 binding
var ActiveY binary;                     # 1 = bound Y >= 0 binding

minimize LeaderCost: X + 2 * Y;         # leader objective
maximize FollowerValue: Y;              # used only to check the response

# Follower primal feasibility.
subject to FollowerLimit1: Y <= 1 + X;
subject to FollowerLimit2: Y <= 6 - X;  # second follower limit

# Stationarity of the follower Lagrangian for min -Y.
subject to Stationarity: Mu1 + Mu2 - Nu = 1;

# Complementarity, linearized with binaries and valid bounds.
subject to Pair1Dual:  Mu1 <= dual_bound * Active1;
subject to Pair1Slack: 1 + X - Y <= slack_bound * (1 - Active1);  # limit 1 tight when Active1 = 1
subject to Pair2Dual:  Mu2 <= dual_bound * Active2;  # Mu2 = 0 unless Active2 = 1
subject to Pair2Slack: 6 - X - Y <= slack_bound * (1 - Active2);  # limit 2 tight when Active2 = 1
subject to PairYDual:  Nu <= dual_bound * ActiveY;  # Nu = 0 unless ActiveY = 1
subject to PairYSlack: Y <= slack_bound * (1 - ActiveY);  # Y = 0 when ActiveY = 1

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["dual_bound"] = 10
ampl.param["slack_bound"] = 7

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, round(ampl.get_value("X"), 4), round(ampl.get_value("Y"), 4),
      round(ampl.obj["LeaderCost"].value(), 4), round(ampl.var["Mu1"].value(), 4))

ampl.param["dual_bound"] = 0.5          # too small: excludes the true multiplier 1
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.get_value("X"), ampl.get_value("Y"),
      ampl.obj["LeaderCost"].value())

# Check the first response: fix X = 0 and solve the follower alone.
ampl.var["X"].fix(0)
ampl.eval("problem Follower: Y, FollowerLimit1, FollowerLimit2, FollowerValue;  # follower only")
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.get_value("Y"))

solved 0 1.0 2.0 1.0
solved 2.5 3.5 9.5
solved 1


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param dual_bound := 10;
param slack_bound := 7;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display X, Y, LeaderCost, Mu1;
let dual_bound := 0.5;
solve;
display X, Y, LeaderCost;
fix X := 0;
# follower alone, to check its response
problem Follower: Y, FollowerLimit1, FollowerLimit2, FollowerValue;
solve;
display Y;

HiGHS 1.14.0: optimal solution; objective 1.999999
0 simplex iterations
1 branching nodes
Objective = LeaderCost
X = 0
Y = 1
LeaderCost = 2
Mu1 = 1

HiGHS 1.14.0: optimal solution; objective 9.5
0 simplex iterations
0 barrier iterations
Objective = LeaderCost
X = 2.5
Y = 3.5
LeaderCost = 9.5

HiGHS 1.14.0: optimal solution; objective 1
0 simplex iterations
0 barrier iterations
Y = 1

